# Predittore PEGI da Steam Tags — Versione CORRETTA

### Fix rispetto alla versione precedente

| Problema nella v. precedente | Fix in questa versione |
|---|---|
| Valutazione finale fatta sui dati di training (post-SMOTE) → metriche gonfiate (F1 0.80 falso) | **Split train/test stratificato PRIMA di tutto**; il test set non viene mai toccato da feature selection o SMOTE |
| Feature selection (`mutual_info_classif`) fatta su tutto il dataset | Fit **solo su X_train** |
| BorderlineSMOTE su feature binarie (tag 0/1) → genera valori frazionari senza senso | **SMOTENC**, che tratta correttamente le feature categoriche (tag/genere/piattaforma) e quelle continue (prezzo, recensioni, anno) |
| Solo i tag usati come feature | Aggiunte **generi** (separati dai tag), **feature numeriche** (prezzo, anno, rating, numero recensioni, lingue supportate, RAM) e **overall_reviews codificato in modo ordinale** |
| Solo XGBoost e LightGBM tunati con Optuna | **Tutti i 4 base-learner tunati** (XGBoost, LightGBM, CatBoost, ExtraTrees) |
| Nessun confronto train vs test | Riportate **entrambe le metriche** per verificare che il gap di leakage sia chiuso |

L'architettura di stacking (4 base learner → meta-learner LogReg) resta la stessa: era già corretta, il problema non era lì.


In [ ]:
# ── Setup e caricamento dataset ─────────────────────────────────────
import pandas as pd
import numpy as np
import os
import ast
import re
import warnings
warnings.filterwarnings('ignore')

print('[INFO] Sezione 1: Caricamento Dati')
print('=' * 70)

csv_path = None
for candidate in ['/content/for_EDA.csv', 'for_EDA.csv']:
    if os.path.exists(candidate):
        csv_path = candidate
        break
if csv_path is None:
    raise FileNotFoundError('[ERROR] for_EDA.csv non trovato!')

df_raw = pd.read_csv(csv_path)
print(f'[INFO] Dataset caricato: {df_raw.shape[0]} righe × {df_raw.shape[1]} colonne')


## Fase 1 — Feature Engineering (tag + generi + numeriche)

Rispetto alla versione precedente, qui usiamo **tutta** l'informazione disponibile nel dataset, non solo i tag:
- `tags` e `genre` → multi-hot encoding separati (`Tag_*`, `Genre_*`)
- `price`, `year`, `user_rating_all/recent`, `total_review_all/recent`, `supported_language` → feature numeriche continue
- `ram` → estratto il valore numerico in GB da stringhe come `"6 GB"`
- `overall_reviews` → codificato come score **ordinale** (Overwhelmingly Negative=0 → Overwhelmingly Positive=8), non come categoria nominale, perché ha un ordine intrinseco
- `windows`, `mac`, `linux`, `VR` → già binarie, trattate come categoriche
- `title`, `developer`, `publisher`, `month` → **escluse** (identificatori ad alta cardinalità che causerebbero overfitting/memorizzazione, non generalizzano)


In [ ]:
# ── Helper: parsing liste (tag/genere), RAM, overall_reviews ──────────
def clean_list_field(raw):
    """Converte stringhe tipo \"['Action', 'Adventure']\" in liste Python."""
    if pd.isna(raw) or not isinstance(raw, str):
        return []
    raw = raw.strip()
    if raw.startswith('[') and raw.endswith(']'):
        try:
            return ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            pass
    return [x.strip() for x in raw.split(',') if x.strip()]


def parse_ram_gb(raw):
    """Estrae il valore numerico in GB da stringhe come '6 GB', '512 MB'."""
    if pd.isna(raw) or not isinstance(raw, str):
        return np.nan
    m = re.search(r'([\d.]+)\s*(GB|MB)', raw, re.IGNORECASE)
    if not m:
        return np.nan
    val, unit = float(m.group(1)), m.group(2).upper()
    return val / 1024 if unit == 'MB' else val


# Mappatura ordinale Steam review score (intrinsecamente ordinata, non nominale)
OVERALL_REVIEWS_MAP = {
    'overwhelmingly negative': 0, 'very negative': 1, 'negative': 2,
    'mostly negative': 3, 'mixed': 4, 'mostly positive': 5,
    'positive': 6, 'very positive': 7, 'overwhelmingly positive': 8,
}


def map_overall_reviews(raw):
    if pd.isna(raw) or not isinstance(raw, str):
        return np.nan
    return OVERALL_REVIEWS_MAP.get(raw.strip().lower(), np.nan)


print('[INFO] Funzioni helper definite.')


In [ ]:
# ── Costruzione feature: tag, generi, numeriche ────────────────────────
from sklearn.preprocessing import MultiLabelBinarizer

# Multi-hot encoding TAG
df_raw['tags_cleaned'] = df_raw['tags'].apply(clean_list_field)
mlb_tags = MultiLabelBinarizer()
tags_enc = mlb_tags.fit_transform(df_raw['tags_cleaned'])
tags_cols = [f"Tag_{g.replace(' ','_').replace('-','_').replace('/','_')}" for g in mlb_tags.classes_]
tags_df = pd.DataFrame(tags_enc, columns=tags_cols, index=df_raw.index)
print(f'[INFO] Tag unici codificati: {len(mlb_tags.classes_)}')

# Multi-hot encoding GENERE (separato dai tag, taxonomy diversa/più pulita)
df_raw['genre_cleaned'] = df_raw['genre'].apply(clean_list_field)
mlb_genre = MultiLabelBinarizer()
genre_enc = mlb_genre.fit_transform(df_raw['genre_cleaned'])
genre_cols = [f"Genre_{g.replace(' ','_').replace('-','_').replace('/','_')}" for g in mlb_genre.classes_]
genre_df = pd.DataFrame(genre_enc, columns=genre_cols, index=df_raw.index)
print(f'[INFO] Generi unici codificati: {len(mlb_genre.classes_)}')

# Feature numeriche continue
df_raw['ram_gb'] = df_raw['ram'].apply(parse_ram_gb)
df_raw['overall_reviews_score'] = df_raw['overall_reviews'].apply(map_overall_reviews)

NUMERIC_COLS = ['price', 'year', 'user_rating_all', 'total_review_all',
                'user_rating_recent', 'total_review_recent',
                'supported_language', 'ram_gb', 'overall_reviews_score']
numeric_df = df_raw[NUMERIC_COLS].copy()

# Feature binarie piattaforma (già 0/1, trattate come categoriche)
PLATFORM_COLS = ['windows', 'mac', 'linux', 'VR']
platform_df = df_raw[PLATFORM_COLS].fillna(0).astype(int)

# Assembla feature set completo
df_encoded = pd.concat([tags_df, genre_df, numeric_df, platform_df], axis=1)

print(f'[INFO] Shape feature set completo: {df_encoded.shape}')
print(f'[INFO]  - Tag:      {len(tags_cols)}')
print(f'[INFO]  - Generi:   {len(genre_cols)}')
print(f'[INFO]  - Numeriche: {len(NUMERIC_COLS)}')
print(f'[INFO]  - Piattaforma: {len(PLATFORM_COLS)}')


In [ ]:
# ── Mappatura target PEGI → classi [0-4] ───────────────────────────────
pegi_col = None
for candidate in ['pegi', 'required_age', 'age_rating', 'rating']:
    if candidate in df_raw.columns:
        pegi_col = candidate
        break
if pegi_col is None:
    raise ValueError('[ERROR] Impossibile trovare colonna target per PEGI')

PEGI_MAPPING = {3: 0, 7: 1, 12: 2, 16: 3, 18: 4}
PEGI_INV_MAP = {v: k for k, v in PEGI_MAPPING.items()}

df_encoded['target_class'] = df_raw[pegi_col].map(PEGI_MAPPING)
df_final = df_encoded.dropna(subset=['target_class']).copy()
df_final['target_class'] = df_final['target_class'].astype(int)

print(f'[INFO] Colonna target: "{pegi_col}"')
print(f'[INFO] Righe scartate per target mancante: {len(df_encoded) - len(df_final)} '
      f'({(len(df_encoded)-len(df_final))/len(df_encoded)*100:.1f}%)')
print('\n[INFO] Distribuzione classi:')
for pv, ci in sorted(PEGI_MAPPING.items()):
    count = (df_final['target_class'] == ci).sum()
    pct = count / len(df_final) * 100
    print(f'   PEGI {pv:>2} (Classe {ci}): {count:5d} campioni ({pct:5.1f}%)')
print(f'\n[INFO] Shape dataset finale: {df_final.shape}')

# NOTA: la perdita di righe per target mancante resta un limite del dataset,
# non risolvibile a livello di codice. Se possibile, vale la pena indagare
# se altre colonne (es. 'pegi_rated') permettono di recuperare più righe.


## Fase 2 — Split Train/Test (PRIMA di feature selection e bilanciamento)

Questo è il fix più importante. Lo split avviene **ora**, prima di qualsiasi fitting (`SelectKBest`, `SMOTENC`, modelli). Il test set non sarà toccato fino alla valutazione finale.


In [ ]:
# ── Train/Test Split stratificato (80/20) ───────────────────────────────
from sklearn.model_selection import train_test_split

X_full = df_final.drop(columns=['target_class']).fillna(0)  # safety net su eventuali NaN residui nei tag/generi
y_full = df_final['target_class'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=42
)

print(f'[INFO] X_train: {X_train_raw.shape}  |  X_test: {X_test_raw.shape}')
print('\n[INFO] Distribuzione classi nel TRAIN:')
for ci in sorted(np.unique(y_train)):
    pct = (y_train == ci).sum() / len(y_train) * 100
    print(f'   Classe {ci} (PEGI {PEGI_INV_MAP[ci]}): {(y_train==ci).sum():5d} ({pct:5.1f}%)')
print('\n[INFO] Distribuzione classi nel TEST (mai toccato da ora in poi):')
for ci in sorted(np.unique(y_test)):
    pct = (y_test == ci).sum() / len(y_test) * 100
    print(f'   Classe {ci} (PEGI {PEGI_INV_MAP[ci]}): {(y_test==ci).sum():5d} ({pct:5.1f}%)')


## Fase 3 — Imputazione numerica + Feature Selection (fit SOLO su train)

`SelectKBest` con `mutual_info_classif` viene addestrato esclusivamente su `X_train`. `X_test` viene solo trasformato con il selettore già fit, mai usato per scegliere le feature.


In [ ]:
# ── Imputazione numerica (fit su train, transform su train e test) ─────
from sklearn.impute import SimpleImputer

def is_categorical_col(colname):
    """Vero per tag, generi e flag piattaforma (feature binarie/nominali)."""
    return colname.startswith('Tag_') or colname.startswith('Genre_') or colname in PLATFORM_COLS

feature_cols_full = X_train_raw.columns.tolist()
categorical_mask_full = np.array([is_categorical_col(c) for c in feature_cols_full])
numeric_cols_full = [c for c in feature_cols_full if not is_categorical_col(c)]

num_imputer = SimpleImputer(strategy='median')
X_train_raw[numeric_cols_full] = num_imputer.fit_transform(X_train_raw[numeric_cols_full])
X_test_raw[numeric_cols_full]  = num_imputer.transform(X_test_raw[numeric_cols_full])

print(f'[INFO] Feature numeriche imputate (mediana, fit solo su train): {len(numeric_cols_full)}')
print(f'[INFO] Feature categoriche (tag+genere+piattaforma): {categorical_mask_full.sum()}')


In [ ]:
# ── Feature Selection con Mutual Information (fit SOLO su X_train) ─────
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import joblib

K_FEATURES = min(200, X_train_raw.shape[1])
print(f'[INFO] Selezione top-{K_FEATURES} feature su {X_train_raw.shape[1]} disponibili...')

# discrete_features specifica quali colonne sono categoriche per il calcolo
# corretto della mutual information (essenziale con feature mix categoriche+continue)
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(
        X, y, discrete_features=categorical_mask_full, random_state=42
    ),
    k=K_FEATURES
)

X_train_sel_arr = selector.fit_transform(X_train_raw, y_train)   # fit SOLO su train
X_test_sel_arr  = selector.transform(X_test_raw)                  # solo transform su test

sel_names = X_train_raw.columns[selector.get_support()].tolist()
X_train_sel = pd.DataFrame(X_train_sel_arr, columns=sel_names, index=X_train_raw.index)
X_test_sel  = pd.DataFrame(X_test_sel_arr,  columns=sel_names, index=X_test_raw.index)

# Maschera categorica ricalcolata sulle sole feature selezionate (serve per SMOTENC)
categorical_mask_sel = np.array([is_categorical_col(c) for c in sel_names])

n_tags   = sum(1 for c in sel_names if c.startswith('Tag_'))
n_genres = sum(1 for c in sel_names if c.startswith('Genre_'))
n_numeric_sel = sum(1 for c in sel_names if not is_categorical_col(c))
n_platform = sum(1 for c in sel_names if c in PLATFORM_COLS)

print(f'[INFO] Feature selezionate: {len(sel_names)}')
print(f'[INFO]  - Tag:        {n_tags}')
print(f'[INFO]  - Generi:     {n_genres}')
print(f'[INFO]  - Numeriche:  {n_numeric_sel}')
print(f'[INFO]  - Piattaforma:{n_platform}')

joblib.dump(selector, 'feature_selector.pkl')
joblib.dump(sel_names, 'selected_features.pkl')
joblib.dump(num_imputer, 'numeric_imputer.pkl')
joblib.dump(categorical_mask_sel, 'categorical_mask.pkl')


## Fase 4 — Setup CV e Bilanciamento (SMOTENC al posto di SMOTE su feature binarie)

`SMOTENC` interpola correttamente le feature continue (prezzo, anno, recensioni) e gestisce quelle categoriche (tag, generi, piattaforma) per moda, evitando i valori frazionari senza senso che il vecchio BorderlineSMOTE generava sui tag binari.


In [ ]:
!pip install -q optuna xgboost lightgbm catboost imbalanced-learn shap scikit-learn


In [ ]:
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTENC

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Indici delle colonne categoriche tra le feature selezionate (richiesti da SMOTENC)
cat_feature_indices = np.where(categorical_mask_sel)[0].tolist()

def make_smotenc():
    return SMOTENC(
        categorical_features=cat_feature_indices,
        random_state=42, k_neighbors=5
    )

print(f'[INFO] StratifiedKFold: {N_SPLITS} fold (eseguito SOLO su X_train_sel/y_train)')
print(f'[INFO] SMOTENC configurato con {len(cat_feature_indices)} feature categoriche su {len(sel_names)}')


## Fase 5 — Hyperparameter Tuning (tutti i 4 base-learner, non solo 2)

In ogni fold di CV, `SMOTENC` viene applicato **solo al fold di train**; la valutazione è sul fold di validazione mai bilanciato. Questo dà una stima onesta della F1 macro reale, senza alcun leakage — è la stessa logica corretta già usata in precedenza per XGB/LGB, qui estesa anche a CatBoost ed ExtraTrees.


In [ ]:
# ── 5a: Ottimizzazione XGBoost ──────────────────────────────────────────
print('[INFO] Tuning XGBoost (30 trial)...')

def objective_xgb(trial):
    params = {
        'objective': 'multi:softprob', 'num_class': 5, 'eval_metric': 'mlogloss',
        'random_state': 42, 'n_jobs': -1,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        try:
            Xtr_r, ytr_r = make_smotenc().fit_resample(Xtr, ytr)
        except Exception:
            Xtr_r, ytr_r = Xtr, ytr
        m = xgb.XGBClassifier(**params)
        m.fit(Xtr_r, ytr_r, verbose=False)
        scores.append(f1_score(yvl, m.predict(Xvl), average='macro'))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
print(f'\n[XGBoost] F1 Macro (CV, onesta): {study_xgb.best_value:.4f}')
for k, v in study_xgb.best_params.items():
    print(f'   {k}: {v}')


In [ ]:
# ── 5b: Ottimizzazione LightGBM ─────────────────────────────────────────
print('[INFO] Tuning LightGBM (20 trial)...')

def objective_lgb(trial):
    params = {
        'objective': 'multiclass', 'num_class': 5, 'random_state': 42,
        'n_jobs': -1, 'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        try:
            Xtr_r, ytr_r = make_smotenc().fit_resample(Xtr, ytr)
        except Exception:
            Xtr_r, ytr_r = Xtr, ytr
        m = lgb.LGBMClassifier(**params)
        m.fit(Xtr_r, ytr_r)
        scores.append(f1_score(yvl, m.predict(Xvl), average='macro'))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=True)
print(f'\n[LightGBM] F1 Macro (CV, onesta): {study_lgb.best_value:.4f}')
for k, v in study_lgb.best_params.items():
    print(f'   {k}: {v}')


In [ ]:
# ── 5c: Ottimizzazione CatBoost (NUOVO — non tunato nella v. precedente) ──
print('[INFO] Tuning CatBoost (15 trial)...')

def objective_cat(trial):
    params = {
        'loss_function': 'MultiClass', 'random_state': 42,
        'thread_count': -1, 'verbose': 0,
        'iterations': trial.suggest_int('iterations', 150, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        try:
            Xtr_r, ytr_r = make_smotenc().fit_resample(Xtr, ytr)
        except Exception:
            Xtr_r, ytr_r = Xtr, ytr
        m = CatBoostClassifier(**params)
        m.fit(Xtr_r, ytr_r)
        scores.append(f1_score(yvl, m.predict(Xvl), average='macro'))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3))
study_cat.optimize(objective_cat, n_trials=15, show_progress_bar=True)
print(f'\n[CatBoost] F1 Macro (CV, onesta): {study_cat.best_value:.4f}')
for k, v in study_cat.best_params.items():
    print(f'   {k}: {v}')


In [ ]:
# ── 5d: Ottimizzazione ExtraTrees (NUOVO — non tunato nella v. precedente) ─
print('[INFO] Tuning ExtraTrees (15 trial)...')

def objective_et(trial):
    params = {
        'random_state': 42, 'n_jobs': -1, 'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }
    scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        try:
            Xtr_r, ytr_r = make_smotenc().fit_resample(Xtr, ytr)
        except Exception:
            Xtr_r, ytr_r = Xtr, ytr
        m = ExtraTreesClassifier(**params)
        m.fit(Xtr_r, ytr_r)
        scores.append(f1_score(yvl, m.predict(Xvl), average='macro'))
    return np.mean(scores)

study_et = optuna.create_study(direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3))
study_et.optimize(objective_et, n_trials=15, show_progress_bar=True)
print(f'\n[ExtraTrees] F1 Macro (CV, onesta): {study_et.best_value:.4f}')
for k, v in study_et.best_params.items():
    print(f'   {k}: {v}')


## Fase 6 — Addestramento Finale dello Stacking Ensemble

`SMOTENC` viene applicato **una sola volta**, solo su `X_train_sel`/`y_train`. `X_test_sel` resta intatto fino alla cella di valutazione.


In [ ]:
# ── Costruzione base-learner con i migliori iperparametri trovati ───────
best_xgb_p = {**study_xgb.best_params, 'objective':'multi:softprob','num_class':5,
              'eval_metric':'mlogloss','random_state':42,'n_jobs':-1}
best_lgb_p = {**study_lgb.best_params, 'objective':'multiclass','num_class':5,
              'random_state':42,'n_jobs':-1,'verbose':-1}
best_cat_p = {**study_cat.best_params, 'loss_function':'MultiClass','random_state':42,
              'thread_count':-1,'verbose':0}
best_et_p  = {**study_et.best_params, 'random_state':42,'n_jobs':-1,'class_weight':'balanced'}

xgb_final = xgb.XGBClassifier(**best_xgb_p)
lgb_final = lgb.LGBMClassifier(**best_lgb_p)
cat_final = CatBoostClassifier(**best_cat_p)
et_final  = ExtraTreesClassifier(**best_et_p)

meta_learner = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, C=1.0, multi_class='multinomial', random_state=42)
)

stacking_clf = StackingClassifier(
    estimators=[('xgb', xgb_final), ('lgb', lgb_final), ('cat', cat_final), ('et', et_final)],
    final_estimator=meta_learner,
    cv=5, stack_method='predict_proba', n_jobs=-1, passthrough=False
)

# Bilanciamento SOLO del training set
print('[INFO] Applicando SMOTENC al training set...')
X_train_res, y_train_res = make_smotenc().fit_resample(X_train_sel, y_train)
print(f'[INFO] Train: {X_train_sel.shape} → bilanciato: {X_train_res.shape}')

print('[INFO] Addestramento StackingClassifier sul training set bilanciato...')
stacking_clf.fit(X_train_res, y_train_res)
print('[INFO] Addestramento completato.')

joblib.dump(stacking_clf, 'stacking_pegi_model.pkl')
joblib.dump(sel_names, 'model_features.pkl')
print("[INFO] Modello salvato in 'stacking_pegi_model.pkl'")


## Fase 7 — Valutazione (SOLO sul test set, mai usato finora)

Riportiamo sia le metriche su `X_train_sel` (originale, **non** bilanciato) sia su `X_test_sel` per verificare il gap train→test: deve essere piccolo. Se train ≫ test, c'è ancora overfitting/leakage; se sono vicini, la valutazione è finalmente attendibile.


In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    matthews_corrcoef, balanced_accuracy_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns

pegi_labels = [f'PEGI {PEGI_INV_MAP[c]}' for c in range(5)]

y_pred_train = stacking_clf.predict(X_train_sel)   # train ORIGINALE (non risampionato)
y_pred_test  = stacking_clf.predict(X_test_sel)     # test MAI toccato finora
y_proba_test = stacking_clf.predict_proba(X_test_sel)

f1_train = f1_score(y_train, y_pred_train, average='macro')
f1_test  = f1_score(y_test,  y_pred_test,  average='macro')

print('='*70)
print(' CONFRONTO TRAIN vs TEST (verifica assenza di leakage)')
print('='*70)
print(f' F1 Macro TRAIN (originale, non bilanciato): {f1_train:.4f}')
print(f' F1 Macro TEST  (mai visto dal modello):     {f1_test:.4f}')
print(f' Gap (train - test): {f1_train - f1_test:+.4f}  '
      f'{"→ overfitting significativo" if f1_train - f1_test > 0.10 else "→ gap contenuto, OK"}')
print('='*70)

print('\n[REPORT SUL TEST SET]')
print(classification_report(y_test, y_pred_test, target_names=pegi_labels))


In [ ]:
# ── Metriche avanzate sul TEST SET ───────────────────────────────────────
kappa   = cohen_kappa_score(y_test, y_pred_test)
mcc     = matthews_corrcoef(y_test, y_pred_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_test)
roc_auc = roc_auc_score(y_test, y_proba_test, multi_class='ovr', average='macro')

print('[METRICHE AVANZATE — TEST SET]')
print(f' Cohen Kappa:          {kappa:.4f}')
print(f' Matthews Corr. Coef.: {mcc:.4f}')
print(f' Balanced Accuracy:    {bal_acc:.4f}')
print(f' Macro ROC-AUC (OvR):  {roc_auc:.4f}')
print(f'\n[CONFRONTO con stima CV onesta durante Optuna]')
print(f' XGBoost CV F1:    {study_xgb.best_value:.4f}')
print(f' LightGBM CV F1:   {study_lgb.best_value:.4f}')
print(f' CatBoost CV F1:   {study_cat.best_value:.4f}')
print(f' ExtraTrees CV F1: {study_et.best_value:.4f}')
print(f' Stacking F1 (test reale): {f1_test:.4f}  '
      f'({"miglioramento" if f1_test > max(study_xgb.best_value, study_lgb.best_value, study_cat.best_value, study_et.best_value) else "nessun guadagno netto"} '
      f'rispetto al miglior singolo modello)')


In [ ]:
# ── Matrice di confusione (TEST SET) ─────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_test)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=pegi_labels, yticklabels=pegi_labels,
            cbar_kws={'label': 'Frequenza Predizioni (%)'})
plt.title('Matrice di Confusione Normalizzata — TEST SET (%)', fontsize=13, fontweight='bold')
plt.ylabel('Classe Reale'); plt.xlabel('Classe Predetta')
plt.tight_layout(); plt.show()


In [ ]:
# ── Curve ROC multi-classe (TEST SET) ────────────────────────────────────
y_test_bin = label_binarize(y_test, classes=[0,1,2,3,4])
colors_roc = ['steelblue', 'darkorange', 'green', 'crimson', 'purple']

plt.figure(figsize=(9, 7))
for i, (lab, col) in enumerate(zip(pegi_labels, colors_roc)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_test[:, i])
    roc_i = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=col, lw=2, label=f'{lab} (AUC = {roc_i:.3f})')
plt.plot([0,1],[0,1], 'k--', lw=1.5, label='Classificatore Casuale')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Curve ROC Multi-Classe (OvR) — TEST SET', fontsize=13, fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=.35)
plt.tight_layout(); plt.show()


In [ ]:
# ── Feature Importance (sub-modello XGBoost) ─────────────────────────────
imp_scores = stacking_clf.named_estimators_['xgb'].feature_importances_
imp_df = pd.DataFrame({'Feature': sel_names, 'Importance': imp_scores}).sort_values(
    'Importance', ascending=False)

top_n = min(25, len(imp_df))
plt.figure(figsize=(12, 9))
sns.barplot(data=imp_df.head(top_n), x='Importance', y='Feature',
            hue='Feature', palette='viridis', legend=False)
plt.title(f'Top-{top_n} Feature Importance — Sub-modello XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Contributo per gruppo (tag vs generi vs numeriche vs piattaforma)
groups = {
    'Tag':         [c for c in sel_names if c.startswith('Tag_')],
    'Genere':      [c for c in sel_names if c.startswith('Genre_')],
    'Numeriche':   [c for c in sel_names if not is_categorical_col(c)],
    'Piattaforma': [c for c in sel_names if c in PLATFORM_COLS],
}
total_imp = imp_df['Importance'].sum()
print('\n[CONTRIBUTO PER GRUPPO DI FEATURE — sub-modello XGBoost]')
for name, cols in groups.items():
    grp_imp = imp_df[imp_df['Feature'].isin(cols)]['Importance'].sum()
    pct = grp_imp / total_imp * 100 if total_imp > 0 else 0
    print(f' - {name:<12} {len(cols):>4} feature  →  {pct:5.1f}% importanza totale')


In [ ]:
# ── SHAP — interpretabilità sul TEST SET (non sul train) ────────────────
import shap

print('[INFO] Calcolo SHAP values su un campione del TEST set...')
explainer   = shap.TreeExplainer(stacking_clf.named_estimators_['xgb'])
sample_size = min(300, len(X_test_sel))
X_sample    = X_test_sel.sample(n=sample_size, random_state=42)
shap_values = explainer.shap_values(X_sample)

plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='bar',
                   class_names=pegi_labels, show=False)
plt.title('SHAP Feature Importance per Classe PEGI — campione TEST', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## Riepilogo finale

Confronta `f1_test` qui sopra con il valore F1 ≈ 0.61 della versione precedente (quello onesto, da CV, non quello gonfiato a 0.80). Se `f1_test` è vicino o superiore a quel valore, significa che le feature aggiuntive (generi, numeriche) e/o SMOTENC stanno effettivamente aiutando — e ora puoi fidarti del numero, perché è calcolato su dati che il modello non ha mai visto in nessuna forma.

Limiti che restano e non sono risolvibili solo a livello di codice:
- La perdita di righe per `age_rating` mancante (se la percentuale di scarto è ancora alta, vale la pena indagare la fonte dati per recuperare più campioni).
- Se il numero di classi rare (es. PEGI 7) resta basso anche nel train, le metriche per quella classe specifica restano più instabili indipendentemente da quanto bilanci i dati sinteticamente.
